# 03 — Dependency Injection and Middleware

`Depends()`, shared/sub-dependencies, request-scoped resources (like a DB session), middleware, CORS, and centralized exception handling — the plumbing that turns a handful of route functions into a real application.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*httpx2.*")   # silence a harmless TestClient/httpx notice

from fastapi import FastAPI, Depends, HTTPException, Header, Request
from fastapi.testclient import TestClient
from fastapi.middleware.cors import CORSMiddleware
from starlette.responses import JSONResponse
import time

app = FastAPI()

## 1. `Depends()` — FastAPI's dependency injection

A dependency is just a callable (function or class); `Depends(fn)` tells FastAPI to call `fn` before your route handler and pass its return value in as an argument. This is how FastAPI shares logic across endpoints — auth checks, DB sessions, pagination params — **without decorators or global state**, and each dependency's own parameters are validated exactly like a route's.

In [2]:
def pagination_params(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}

@app.get("/items")
def list_items(pagination: dict = Depends(pagination_params)):
    return {"pagination": pagination}

client = TestClient(app)
print(client.get("/items").json())
print(client.get("/items?skip=20&limit=5").json())

{'pagination': {'skip': 0, 'limit': 10}}
{'pagination': {'skip': 20, 'limit': 5}}


## 2. Auth as a dependency, and raising `HTTPException`

A dependency that reads a header/token and raises `HTTPException` on failure is the standard FastAPI auth pattern — `Depends(get_current_user)` on every protected route reuses the same check, and FastAPI turns the raised exception directly into the HTTP response with no extra handling needed in the route.

In [3]:
def get_current_user(authorization: str | None = Header(default=None)):
    if authorization != "Bearer secret-token":
        raise HTTPException(status_code=401, detail="Not authenticated")
    return {"user_id": 1, "username": "alice"}

@app.get("/me")
def read_current_user(user: dict = Depends(get_current_user)):
    return user

client = TestClient(app)
print(client.get("/me").status_code)   # 401 -- no header
print(client.get("/me", headers={"Authorization": "Bearer secret-token"}).json())

401
{'user_id': 1, 'username': 'alice'}


## 3. Dependencies with `yield` — setup/teardown (the DB session pattern)

A dependency using `yield` instead of `return` runs the code **before** the `yield` prior to the route, and the code **after** it once the response has been sent — the standard shape for "open a DB session, hand it to the route, close it afterward even if the route raised."

In [4]:
session_log = []

def get_db_session():
    session_log.append("opened")
    try:
        yield {"connection": "fake-db-session"}
    finally:
        session_log.append("closed")

@app.get("/records")
def list_records(db: dict = Depends(get_db_session)):
    return {"using": db["connection"]}

client = TestClient(app)
print(client.get("/records").json())
print(session_log)   # ['opened', 'closed'] -- teardown ran even though nothing failed

{'using': 'fake-db-session'}
['opened', 'closed']


## 4. Sub-dependencies

A dependency can itself depend on another via `Depends()` in its own signature — FastAPI resolves the whole chain, and by default **caches** a dependency's result per request if it's needed more than once (so two routes/dependencies both depending on `get_current_user` in the same request call it only once).

In [5]:
def require_admin(user: dict = Depends(get_current_user)):
    if user["username"] != "alice":
        raise HTTPException(status_code=403, detail="Admins only")
    return user

@app.delete("/items/{item_id}")
def delete_item(item_id: int, admin: dict = Depends(require_admin)):
    return {"deleted": item_id, "by": admin["username"]}

client = TestClient(app)
print(client.delete("/items/5", headers={"Authorization": "Bearer secret-token"}).json())

{'deleted': 5, 'by': 'alice'}


## 5. Middleware

Middleware wraps **every** request/response, regardless of route — the right place for cross-cutting concerns that don't belong in any one handler: request timing/logging, adding response headers, or CORS. Contrast with `Depends()`, which is opt-in per route.

**Important ordering constraint:** middleware (and exception handlers, section 6) must be registered **before** the app handles its first request — Starlette builds the middleware stack lazily on first use and locks it in. That's why this cell uses a **fresh** `FastAPI()` instance instead of continuing to add to `app` from the sections above, which has already served requests via `TestClient`.

In [6]:
app_mw = FastAPI()

@app_mw.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    response.headers["X-Process-Time-Ms"] = f"{(time.perf_counter() - start) * 1000:.2f}"
    return response

@app_mw.get("/ping")
def ping():
    return {"pong": True}

mw_client = TestClient(app_mw)   # first request on app_mw -- fine, middleware was added first
resp = mw_client.get("/ping")
print(resp.json())
print(resp.headers["x-process-time-ms"])

{'pong': True}
0.70


`CORSMiddleware` is the built-in example you'll use in almost every real API with a separate frontend origin:

```python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://myfrontend.example.com"],
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)
```

## 6. Centralized exception handling

`@app.exception_handler(SomeException)` catches a specific exception type raised **anywhere** in a route/dependency and converts it into a consistent response shape — avoiding repeating the same `try/except` block in every handler that can raise that error. Same ordering rule as middleware: register the handler before the app's first request, so another fresh app instance here too.

In [7]:
app_err = FastAPI()

class OutOfStockError(Exception):
    def __init__(self, sku: str):
        self.sku = sku

@app_err.exception_handler(OutOfStockError)
async def out_of_stock_handler(request: Request, exc: OutOfStockError):
    return JSONResponse(status_code=409, content={"error": "out_of_stock", "sku": exc.sku})

@app_err.post("/checkout/{sku}")
def checkout(sku: str):
    if sku == "SOLD-OUT-ITEM":
        raise OutOfStockError(sku)
    return {"checked_out": sku}

err_client = TestClient(app_err)   # first request on app_err -- handler was already registered
resp = err_client.post("/checkout/SOLD-OUT-ITEM")
print(resp.status_code, resp.json())

409 {'error': 'out_of_stock', 'sku': 'SOLD-OUT-ITEM'}


## 7. Interview Q&A

1. **"How do you share an auth check across many routes without repeating code?"** — a dependency function raising `HTTPException` on failure, applied via `Depends()` on each protected route (or globally via a router-level `dependencies=[...]`).
2. **"How do you guarantee a DB connection is closed even if the route raises an exception?"** — a `yield`-based dependency: the code after `yield` (ideally in a `finally`) runs during response teardown regardless of whether the route succeeded.
3. **"When would you reach for middleware instead of a dependency?"** — when the logic must apply to *every* request unconditionally (timing, CORS, global logging) rather than being opted into per route.
4. **"Does FastAPI call a dependency once or multiple times if several things in one request depend on it?"** — once per request by default (dependency result is cached for that request) unless you pass `use_cache=False` to `Depends()`.

## Summary

- `Depends()` injects shared logic (auth, pagination, DB sessions) into routes without global state; results are cached per-request.
- `yield`-based dependencies give you setup/teardown — the pattern for anything that must be cleaned up after the response.
- Middleware wraps every request unconditionally; dependencies are opt-in per route.
- `@app.exception_handler(...)` centralizes error-to-response mapping instead of repeating `try/except` in every handler.
- Next: `04_async_and_concurrency.ipynb`.